In [ ]:
# project setup, run first, do not edit except NAME
NAME = "Roman"# <<< your name

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path("/content/boe-group").exists():
        _t = userdata.get("GH_TOKEN")
        subprocess.run(["git", "clone", "-q",
                        f"https://{_t}@github.com/YOUR-ORG/boe-group.git",
                        "/content/boe-group"], check=True)
    ROOT = pathlib.Path("/content/boe-group")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
               check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n> Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

import torch
print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}  "
      f"cuda {torch.cuda.is_available()}  data {DATA}")

In [ ]:
DATA = ROOT / "data" / "sample"          # temporary: no Drive export yet
df = loading.load_topics(DATA)
print(df.shape)
df.head()

In [ ]:
analyst = df[df["speaker_role"] == "analyst"]
pivot = analyst.pivot(index="quarter", columns="topic", values="mean_sentiment")
pivot.plot(marker="o", title="Analyst sentiment by topic")
plt.axhline(0, linewidth=0.8)
plt.show()